
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>



<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif;">

<div style="background: #1B5162; color: white; border-radius: 8px; padding: 28px 32px; text-align: center; position: relative;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">Lesson 02</div>
  <div style="font-size: 24pt; font-weight: 700; line-height: 1.3;">Find Your Data and Create a table</div>
  <div style="font-size: 14pt; margin-top: 12px; opacity: 0.9;">Navigate the catalog hierarchy, locate a raw data file, preview its contents, and create a table from it.</div>
</div>

</div>

## REQUIRED — SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before running this notebook, confirm your compute environment at the top-right of the notebook.

- Click the compute dropdown and select **Serverless** (the default option).
- If you do not see Serverless available, contact your workspace administrator.

**Note:** This notebook was developed and tested on **Serverless compute**. Other compute options may work but are not guaranteed to behave the same.
  </div>
</div>

### Setup
Run the cell below to configure your environment for this lesson. It will ensure you have the appropriate catalog, schema, and data set up to complete the following work.

In [0]:
%run ./Includes/Classroom-Setup-1


<!-- LEARN: Unity Catalog Hierarchy -->
<!-- Template: vertical-layered-stack (adapted to 3 layers) -->

<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">

<div style="font-size: 20pt; font-weight: 700; color: #0b2026; margin-bottom: 6px;">How Databricks Organizes Your Data</div>
<div style="font-size: 14pt; color: #5A6F77; margin-bottom: 24px;">Everything in Databricks lives in a three-level hierarchy within Unity Catalog. Think of it like folders on your computer, except these folders also control who can see and use the data inside them.</div>

<div style="display: flex; align-items: stretch; gap: 16px;">

<!-- Left arrow label -->
<div style="
    writing-mode: vertical-lr;
    transform: rotate(180deg);
    text-align: center;
    font-weight: 700;
    font-size: 14pt;
    color: #618794;
    padding: 0 6px;
    display: flex;
    justify-content: flex-end;
">
&larr; BROAD TO SPECIFIC
</div>

<!-- Stacked layers -->
<div style="flex: 1; display: flex; flex-direction: column; gap: 6px;">

<!-- Catalog -->
<div style="background: #1C3037; color: white; border-radius: 8px 8px 4px 4px; padding: 22px 24px; text-align: center;">
  <div style="font-size: 18pt; font-weight: 700;">Catalog</div>
  <div style="font-size: 14pt; margin-top: 6px; opacity: 0.9;">The top-level container. Groups related schemas together.</div>
</div>

<!-- Schema -->
<div style="background: #2574B5; color: white; border-radius: 4px; padding: 18px 24px; text-align: center;">
  <div style="font-size: 16pt; font-weight: 700;">Schema</div>
  <div style="font-size: 14pt; margin-top: 6px; opacity: 0.9;">A collection of related tables, views, and volumes. Organizes data by project or domain.</div>
</div>

<!-- Tables & Volumes -->
<div style="background: #02A36F; color: white; border-radius: 4px 4px 8px 8px; padding: 18px 24px; text-align: center;">
  <div style="font-size: 16pt; font-weight: 700;">Tables &amp; Volumes</div>
  <div style="font-size: 14pt; margin-top: 6px; opacity: 0.9;">Tables hold structured data (rows and columns). Volumes hold raw files (CSVs, JSON, images).</div>
</div>

</div>

</div>

<!-- Key point callout -->
<div style="margin-top: 20px; padding: 16px 20px; background: #FFF6F4; border: 3px solid #FF5F46; border-radius: 10px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <strong>Your first job as a data engineer:</strong> Find the raw data files in a volume, then turn them into reliable UC tables that everyone on your team can query.
  </div>
</div>

</div>

##### EXPAND FOR ADDITIONAL NOTES

<details>

**Catalog → Schema → Tables/Volumes** is the path you'll use every time you work with data in Databricks. This is the three-level namespace that makes Unity Catalog stronger at governance and security.

- **Catalogs** are the broadest container. In most organizations, you'll have separate catalogs for development, staging, and production, or organized by business unit. In this course, you're working in a pre-configured catalog.
- **Schemas** (also called databases) group related objects together. A schema might hold all the tables for a specific project, team, or data domain. In Databricks, Schemas hold more than just data tables; they also contain volumes, models, and functions.
- **Tables** store structured, queryable data using an open table format. Once data is in a table, anyone with permission can query it using SQL.
- **Volumes** store raw files before they become tables. Think of a volume as a managed folder where you land CSV files, JSON exports, or other data files that need to be processed.

This hierarchy is managed by **Unity Catalog**, which also handles permissions. When you create a table inside a schema, Unity Catalog automatically tracks who created it, when, and who has access.

</details>


### Explore: Find your data in the catalog

Let's start by confirming where we are in the catalog hierarchy, then find the raw data file waiting for us.

**Step 1:** Run the cell below to confirm your current catalog and schema.

In [0]:
%sql
SELECT current_catalog(), current_schema();

**Step 2:** Let's see what's in our schema. Run the cells below to check for any existing tables and volumes.

In [0]:
%sql
SHOW TABLES;

In [0]:
%sql
SHOW VOLUMES;

**Step 3:** List the files inside the volume. You should see CSV files including `employees.csv` and `employees2.csv`.

In [0]:
spark.sql(f"LIST '/Volumes/{my_catalog}/{my_schema}/myfiles/'").display()

**Step 4:** Preview the CSV data without creating a table. The `read_files` function lets you peek at raw file contents directly.

In [0]:
%sql
SELECT * 
FROM read_files('/Volumes/' || my_catalog || '/' || my_schema || '/myfiles/employees.csv')

You should see 4 rows with columns: **ID**, **FirstName**, **Country**, and **Role**. This is raw file data, not stored as a table yet.

You may also notice a column called **`_rescued_data`** with `null` values. Databricks automatically adds this column when reading files. If any rows don't conform to the inferred schema (wrong data types, extra fields, malformed records), those values are captured here instead of being lost or breaking the read. When all your data is clean, it stays `null`. This is one of the ways Databricks keeps your data safe during ingestion.

### Explore: Create a table from the CSV

Now let's turn that raw CSV into a proper table. The `CREATE TABLE AS SELECT` (CTAS) pattern reads the file and writes the result as a new table in one step. By default Databricks creates tables in Delta format.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS employees
AS 
SELECT * 
FROM read_files('/Volumes/' || my_catalog || '/' || my_schema || '/myfiles/employees.csv');

**Step 5:** Verify the table was created. Run the cell below to query it.

In [0]:
%sql
SELECT * 
FROM employees;

Same 4 rows, but now they're stored as a **table** with a transaction log tracking every change. You can also verify this in **Catalog Explorer**: click **Catalog** in the left sidebar, expand your catalog and schema, and you should see the `employees` table listed under **Tables**. 

The transaction log tracking every change is what enables features like time travel, schema enforcement, and ACID guarantees that you'll explore in later lessons.


<!-- Micro-win summary -->

<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">
<div style="margin-top: 10px; padding: 18px 24px; background: #FFF6F4; border: 3px solid #FF5F46; border-radius: 10px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <div style="font-weight: 700; margin-bottom: 8px;">What you just did:</div>
    <ul style="padding-left: 20px; margin: 0;">
      <li>Navigated the Unity Catalog hierarchy (catalog → schema → volume)</li>
      <li>Previewed raw CSV data using <code>read_files</code></li>
      <li>Created your first UC table using <code>CREATE TABLE AS SELECT</code></li>
      <li>Verified the table in both SQL and Catalog Explorer</li>
    </ul>
  </div>
</div>
</div>


&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>